In [ ]:
from dotenv import load_dotenv
import os
import voyageai

load_dotenv()

voyage_api_key = os.getenv("VOYAGE_API_KEY")

if not voyage_api_key:
    raise ValueError("VOYAGE_API_KEY not found in environment.")

client = voyageai.Client(api_key=voyage_api_key)

In [2]:
# Chunk by section
import re


def chunk_by_section(document_text):
    pattern = r"\n## "
    return re.split(pattern, document_text)

In [3]:
# Embedding Generation
def generate_embedding(chunks, model="voyage-3-large", input_type="query"):
    is_list = isinstance(chunks, list)
    input = chunks if is_list else [chunks]
    result = client.embed(input, model=model, input_type=input_type)
    return result.embeddings if is_list else result.embeddings[0]

In [4]:
# VectorIndex implementation
import math
from typing import Optional, Any, List, Dict, Tuple


class VectorIndex:
    def __init__(
        self,
        distance_metric: str = "cosine",
        embedding_fn=None,
    ):
        self.vectors: List[List[float]] = []
        self.documents: List[Dict[str, Any]] = []
        self._vector_dim: Optional[int] = None
        if distance_metric not in ["cosine", "euclidean"]:
            raise ValueError("distance_metric must be 'cosine' or 'euclidean'")
        self._distance_metric = distance_metric
        self._embedding_fn = embedding_fn

    def add_document(self, document: Dict[str, Any]):
        if not self._embedding_fn:
            raise ValueError(
                "Embedding function not provided during initialization."
            )
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError(
                "Document dictionary must contain a 'content' key."
            )

        content = document["content"]
        if not isinstance(content, str):
            raise TypeError("Document 'content' must be a string.")

        vector = self._embedding_fn(content)
        self.add_vector(vector=vector, document=document)

    def search(
        self, query: Any, k: int = 1
    ) -> List[Tuple[Dict[str, Any], float]]:
        if not self.vectors:
            return []

        if isinstance(query, str):
            if not self._embedding_fn:
                raise ValueError(
                    "Embedding function not provided for string query."
                )
            query_vector = self._embedding_fn(query)
        elif isinstance(query, list) and all(
            isinstance(x, (int, float)) for x in query
        ):
            query_vector = query
        else:
            raise TypeError(
                "Query must be either a string or a list of numbers."
            )

        if self._vector_dim is None:
            return []

        if len(query_vector) != self._vector_dim:
            raise ValueError(
                f"Query vector dimension mismatch. Expected {self._vector_dim}, got {len(query_vector)}"
            )

        if k <= 0:
            raise ValueError("k must be a positive integer.")

        if self._distance_metric == "cosine":
            dist_func = self._cosine_distance
        else:
            dist_func = self._euclidean_distance

        distances = []
        for i, stored_vector in enumerate(self.vectors):
            distance = dist_func(query_vector, stored_vector)
            distances.append((distance, self.documents[i]))

        distances.sort(key=lambda item: item[0])

        return [(doc, dist) for dist, doc in distances[:k]]

    def add_vector(self, vector, document: Dict[str, Any]):
        if not isinstance(vector, list) or not all(
            isinstance(x, (int, float)) for x in vector
        ):
            raise TypeError("Vector must be a list of numbers.")
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError(
                "Document dictionary must contain a 'content' key."
            )

        if not self.vectors:
            self._vector_dim = len(vector)
        elif len(vector) != self._vector_dim:
            raise ValueError(
                f"Inconsistent vector dimension. Expected {self._vector_dim}, got {len(vector)}"
            )

        self.vectors.append(list(vector))
        self.documents.append(document)

    def _euclidean_distance(
        self, vec1: List[float], vec2: List[float]
    ) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")
        return math.sqrt(sum((p - q) ** 2 for p, q in zip(vec1, vec2)))

    def _dot_product(self, vec1: List[float], vec2: List[float]) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")
        return sum(p * q for p, q in zip(vec1, vec2))

    def _magnitude(self, vec: List[float]) -> float:
        return math.sqrt(sum(x * x for x in vec))

    def _cosine_distance(self, vec1: List[float], vec2: List[float]) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")

        mag1 = self._magnitude(vec1)
        mag2 = self._magnitude(vec2)

        if mag1 == 0 and mag2 == 0:
            return 0.0
        elif mag1 == 0 or mag2 == 0:
            return 1.0

        dot_prod = self._dot_product(vec1, vec2)
        cosine_similarity = dot_prod / (mag1 * mag2)
        cosine_similarity = max(-1.0, min(1.0, cosine_similarity))

        return 1.0 - cosine_similarity

    def __len__(self) -> int:
        return len(self.vectors)

    def __repr__(self) -> str:
        has_embed_fn = "Yes" if self._embedding_fn else "No"
        return f"VectorIndex(count={len(self)}, dim={self._vector_dim}, metric='{self._distance_metric}', has_embedding_fn='{has_embed_fn}')"

In [7]:
sample_text = """
# Annual Interdisciplinary Research Review

## Executive Summary

This report summarizes key findings and ongoing research efforts across several departments.
The organization has focused on innovation, collaboration, and solving complex problems across disciplines.

## Medical Research

Researchers investigated new diagnostic methods for rare diseases.
The team identified several promising biomarkers and recommended additional clinical testing.

## Software Engineering

The engineering team analyzed recurring system errors and implemented improvements to memory management.
These changes reduced application crashes and improved overall system stability.

## Financial Analysis

Analysts reviewed quarterly performance and identified changes in resource allocation.
The findings will be used to support future planning and investment decisions.

## Cybersecurity

The cybersecurity team investigated several security incidents and strengthened internal monitoring procedures.
New safeguards were introduced to reduce future risks.
"""

In [8]:
text = sample_text

In [9]:
chunks = chunk_by_section(text)

In [10]:
embeddings = generate_embedding(chunks)

In [11]:
store = VectorIndex()

for chunk, embedding in zip(chunks, embeddings):
    store.add_vector(
        vector=embedding,
        document={"content": chunk}
    )

In [12]:
user_question = "What did the software engineering team improve?"

user_embedding = generate_embedding(user_question)

In [13]:
results = store.search(user_embedding, 2)

for doc, distance in results:
    print("Distance:", distance)
    print(doc["content"])
    print()

Distance: 0.3067725721981345
Software Engineering

The engineering team analyzed recurring system errors and implemented improvements to memory management.
These changes reduced application crashes and improved overall system stability.


Distance: 0.5043479217400739
Cybersecurity

The cybersecurity team investigated several security incidents and strengthened internal monitoring procedures.
New safeguards were introduced to reduce future risks.




In [14]:
retrieved_context = "\n\n".join(
    doc["content"] for doc, distance in results
)

print(retrieved_context)

Software Engineering

The engineering team analyzed recurring system errors and implemented improvements to memory management.
These changes reduced application crashes and improved overall system stability.


Cybersecurity

The cybersecurity team investigated several security incidents and strengthened internal monitoring procedures.
New safeguards were introduced to reduce future risks.



In [15]:
from anthropic import Anthropic

claude_client = Anthropic()

prompt = f"""
Answer the user's question using only the context provided.

User question:
{user_question}

Context:
{retrieved_context}
"""

response = claude_client.messages.create(
    model="claude-sonnet-5",
    max_tokens=300,
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(response.content[0].text)

The software engineering team analyzed recurring system errors and implemented improvements to **memory management**. These changes **reduced application crashes** and **improved overall system stability**.
